# BSMM-8730 — Smart Centres REIT Analysis
## Notebook 1A: Data Collection — Bank of Canada Interest Rate

**Author:** Muhammad Ahmad
**Role:** Member C — Bank of Canada Valet API
**Branch:** feature/bankofcanada-interest-rate
**Date:** July 2026

### What this notebook does:
- Connects to the Bank of Canada Valet API
- Downloads the overnight interest rate history (2020 to 2025)
- Saves the data as a CSV file
- Downloads the CSV to your computer

### Why interest rates matter for Smart Centres:
REITs borrow large amounts of money to buy properties.
When the Bank of Canada raises interest rates, borrowing becomes
more expensive and REIT stock prices typically fall.

### Data Source:
- API: Bank of Canada Valet API
- URL: https://www.bankofcanada.ca/valet-api-how-to/
- Series V39079 = Overnight Money Market Financing Rate
- Frequency: Daily

---

## Step 0 — Install Libraries

In [ ]:
!pip install pandas requests --quiet
print("Done!")

## Step 1 — Import Libraries

In [ ]:
import requests
import pandas as pd
import os
from google.colab import files

print("All libraries imported!")

## Step 2 — Define Settings

In [ ]:
START_DATE   = "2020-01-01"
END_DATE     = "2025-12-31"
RATE_SERIES  = "V39079"

RATE_URL = (
    f"https://www.bankofcanada.ca/valet/observations/{RATE_SERIES}/json"
    f"?start_date={START_DATE}&end_date={END_DATE}"
)

print(f"Series     : {RATE_SERIES}")
print(f"Date range : {START_DATE} to {END_DATE}")
print(f"API URL    : {RATE_URL}")

## Step 3 — Create raw_data Folder

In [ ]:
os.makedirs("raw_data", exist_ok=True)
print("Folder raw_data is ready!")

## Step 4 — Call the API

In [ ]:
print("Calling Bank of Canada API for interest rate data...")

response = requests.get(RATE_URL)

if response.status_code == 200:
    print(f"Success! Status code: {response.status_code}")
else:
    print(f"Error. Status code: {response.status_code}")
    print("Check your internet connection and try again.")

## Step 5 — Parse the Response

In [ ]:
data         = response.json()
observations = data["observations"]

print(f"Total observations received: {len(observations)}")
print()
print("Example raw observation:")
print(observations[0])

In [ ]:
# Extract date and rate value from each observation
rows = []
for obs in observations:
    date  = obs["date"]
    value = obs.get(RATE_SERIES, {}).get("v", None)
    rows.append({"date": date, "overnight_rate_pct": value})

# Convert to DataFrame
df = pd.DataFrame(rows)

# Fix data types
df["date"]               = pd.to_datetime(df["date"])
df["overnight_rate_pct"] = pd.to_numeric(df["overnight_rate_pct"], errors="coerce")

print(f"Rows extracted: {len(df)}")
print()
print(df.head(10).to_string(index=False))

## Step 6 — Validate the Data

In [ ]:
print("=== Interest Rate Data Summary ===")
print(f"  Date range    : {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  Total rows    : {len(df)}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Lowest rate   : {df['overnight_rate_pct'].min()}%")
print(f"  Highest rate  : {df['overnight_rate_pct'].max()}%")
print(f"  Average rate  : {df['overnight_rate_pct'].mean():.2f}%")

## Step 7 — Save as CSV

In [ ]:
FILENAME = "raw_data/bank_of_canada_overnight_rate.csv"
df.to_csv(FILENAME, index=False)

print(f"Saved: {FILENAME}")
print(f"Rows : {len(df)}")

## Step 8 — Download to Your Computer

In [ ]:
files.download(FILENAME)

print("Downloaded!")
print()
print("Next steps:")
print("  1. Go to GitHub repo: 8730-assignment")
print("  2. Switch to branch : feature/bankofcanada-interest-rate")
print("  3. Upload this notebook and the CSV to raw_data/ folder")
print("  4. Commit message   : Add Bank of Canada interest rate data")
print("  5. Open PR 1")